In [1]:
# importing sys
import sys
# This is how you import modules that exist outside of working directory 
# adding mosaique to the system path
sys.path.insert(1, '/workspaces/QML-QPF/mosaiQue') 
sys.path.insert(1, '/workspaces/QML-QPF/Quantifying_Entanglement')

import mosaique as mq
import scipy
from concurrent.futures import ProcessPoolExecutor, as_completed
import itertools
import numpy as np
import pennylane as qml
import os
from mosaique.models.operation import OperationLayer
from tensorflow import keras
import tensorflow as tf 
import time
from Entropy import compute_entropy_for_pair_batch

2026-02-08 09:33:02.821768: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-08 09:33:02.870933: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
# if the below cell raise this error [ModuleNotFoundError: No module named 'tensorrt']
# uncomment and run the next line, then restart the kernel 
# pip install tensorrt

## Check GPU installation is working

In [2]:
import tensorrt as trt

print(f' tensorflow version {tf.__version__}')

print(f' tensorrt version {trt.__version__}')

print(tf.config.list_physical_devices('GPU'))

 tensorflow version 2.15.0
 tensorrt version 10.12.0.36
[]


In [3]:

# Set the environment for asynchronous GPU usage
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

In [4]:
def operation():
    dev = qml.device("default.qubit.tf", wires=4)
    @qml.qnode(dev, interface='tf')
    def cnot(inputs):
        inputs = inputs * np.pi
        qml.AngleEmbedding(inputs[:,...], wires=range(4), rotation='Y')

        qml.CNOT(wires=[0, 1])
        qml.CNOT(wires=[2, 3])

        # Measurement producing 4 classical output values
        return [qml.expval(qml.PauliZ(j)) for j in range(4)]
    return cnot


In [2]:
def preset():
    # Import the images
    from medmnist import BreastMNIST
    # size of the images 
    size = 28
    # we'll run the code for 64 to see if we'll get different results
    train_data = BreastMNIST(split="train", download=True,size=size)
    test_data = BreastMNIST(split="test", download=True,size=size)
    # name of the folder where will save the filtered images for training
    
    # name of the folder where will save the filtered images for training
    tr_layer = mq.ConvolutionLayer4x4("_OA_results_BreastMNIST_train_with_val")
    # name of the folder where will save the filtered images for testing
    te_layer = mq.ConvolutionLayer4x4("_OA_results_BreastMNIST_test_with_val")
    
    
        
    #load the data set want to train
    (tr_images, tr_labels), (te_images, te_labels) = (train_data.imgs, train_data.labels), (test_data.imgs, test_data.labels)
    

    tr_layer.fit(tr_images)
    te_layer.fit(te_images)
    tr_images = tr_layer.transform(tr_images)
    te_images = te_layer.transform(te_images)
    return ((tr_layer, (tr_images, tr_labels)), (te_layer,(te_images, te_labels)))

with ProcessPoolExecutor() as executor:
    future = executor.submit(preset)

((train_layer, (train_images, train_labels)), (test_layer,(test_images, test_labels))) = future.result() #This blocks until the task completes

In [3]:

permutations = np.asarray(list(itertools.permutations(range(4))))

def pool(x, call, p, l):
    op = OperationLayer(call())
    predict = l.post_transform(op.pre_op.predict(x,batch_size=1000))
    l.save(predict, p)

In [ ]:
# # not using PorcessPoolExecutor

# for p in permutations:
#     pool(train_images[:,:,p],)

In [7]:
for j in range(3):
    with ProcessPoolExecutor(8) as executor:
        runner = {
            executor.submit(pool, x=train_images[:,:,p], call=operation, p=p, l=train_layer): p for p in permutations[8*j:8*(j+1)]
        }
        for future in as_completed(runner):
            runner.pop(future)

1/1 [==============================] - 2s 2s/step


In [8]:
for j in range(3):
    with ProcessPoolExecutor(8) as executor:
        runner = {
            executor.submit(pool, x=test_images[:,:,p], call=operation, p=p, l=test_layer): p for p in permutations[8*j:8*(j+1)]
        }
        for future in as_completed(runner):
            runner.pop(future)

1/1 [==============================] - 2s 2s/step


## Filtering Data by Entropy degree

In [9]:
# --- Function to process a single input type ---
def process_input_type(input_data, config, entropy_dict_for_symm):
    """
    Helper function to encapsulate the parallel processing logic for one input type.
    """
    input_system_03 = input_data[:, :, :2]
    input_system_21 = input_data[:, :, 2:4] # Assuming the same slicing  input_system_03 = input_data[:, :, :2]

    task_inputs = list(zip(input_system_03, input_system_21))
    # print("Im inside process_input_type")
    # print("len of task inputs is ", len(task_inputs) )
    # print(f"\nStarting parallel entropy computation for {data_name} data ({len(task_inputs)} batches)...")
    type_start_time = time.time()
    # from collections import defaultdict

    # entropy_dict_for_symm = defaultdict(list)   # missing keys become []
    # config = ''.join(map(str, config)) 
    with ProcessPoolExecutor(max_workers=None) as executor:
        results_iterator = executor.map(compute_entropy_for_pair_batch, task_inputs)

        for i, result in enumerate(results_iterator):
            # print(f"i is {i} and result is {type(result)} len {len(result)}")
            (avg_s1_rho0, avg_s1_rho1), \
            (avg_s2_rho0, avg_s2_rho1), \
            (max_s1_rho0, max_s1_rho1), \
            (max_s2_rho0, max_s2_rho1), \
            (S1_rho_0), \
            (S1_rho_1), \
            (S2_rho_0), \
            (S2_rho_1) = result
            
            # Append to the lists directly from the dictionary
            # Using dictionary keys here
            entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S1'].append((avg_s1_rho0, avg_s1_rho1))
            entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S2'].append((avg_s2_rho0, avg_s2_rho1))
            entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S1'].append((max_s1_rho0, max_s1_rho1))
            entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S2'].append((max_s2_rho0, max_s2_rho1))
            # New ones to capture pair entanglement for subsequent filtering
            entropy_dict_for_symm['S1_rho_0_' + config + '_Entropy_S1_0'].append(S1_rho_0)
            entropy_dict_for_symm['S1_rho_1_' + config + '_Entropy_S1_1'].append(S1_rho_1)
            entropy_dict_for_symm['S2_rho_0_' + config + '_Entropy_S2_0'].append(S2_rho_0)
            entropy_dict_for_symm['S2_rho_1_' + config + '_Entropy_S2_1'].append(S2_rho_1)

            if (i + 1) % 100 == 0:
                print(f"Processed {i + 1}/{len(task_inputs)} batches for {config}.")
                
        
        # Convert lists to NumPy arrays
        entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S1'] = np.array(entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S1'])
        entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S2'] = np.array(entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S2'])
        entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S1'] = np.array(entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S1'])
        entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S2'] = np.array(entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S2'])
        # New ones to capture pair entanglement for subsequent filtering
        entropy_dict_for_symm['S1_rho_0_' + config + '_Entropy_S1_0']  = np.array(entropy_dict_for_symm['S1_rho_0_' + config + '_Entropy_S1_0'])    
        entropy_dict_for_symm['S1_rho_1_' + config + '_Entropy_S1_1']  = np.array(entropy_dict_for_symm['S1_rho_1_' + config + '_Entropy_S1_1'])
        entropy_dict_for_symm['S2_rho_0_' + config + '_Entropy_S2_0']  = np.array(entropy_dict_for_symm['S2_rho_0_' + config + '_Entropy_S2_0'])
        entropy_dict_for_symm['S2_rho_1_' + config + '_Entropy_S2_1']  = np.array(entropy_dict_for_symm['S2_rho_1_' + config + '_Entropy_S2_1'])
        
        # compute the total entry of the entire system S1 + S2
        # Compute the total entropy of the entire system S1 + S2
        # Ensure avg_s1_array and avg_s2_array are in the correct shape (N, 2)
        # The output of compute_entropy_for_pair_batch is (rho0, rho1) tuples, which will become (N, 2) arrays.
        entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S1_S2'] = entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S1'] \
                                                                            + entropy_dict_for_symm['Total_Avg_' + config + '_Entropy_S2']
        entropy_dict_for_symm[f'Total_Max_{config}_Entropy_S1_S2'] = entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S1'] \
                                                                            + entropy_dict_for_symm['Total_Max_' + config + '_Entropy_S2']

    # --- Saving Data as CSV ---
    # 1. Create the data_name specific folder

    entropy_folder_name = train_layer.name + f"/{config}/Entropy"
    # train
    # entropy_folder_name = "_OA_train"
    output_folder_path = os.path.join(entropy_folder_name) # Replace spaces for folder names
    os.makedirs(output_folder_path, exist_ok=True)
    print(f"Created/Ensured directory: {output_folder_path}")

    # 2. Save each list/array corresponding to each key as a CSV file
    for key, value in entropy_dict_for_symm.items():
        if isinstance(value, np.ndarray): # Only save if it's a NumPy array
            file_path = os.path.join(output_folder_path, f"{key}.csv")
            # Use fmt='%.8f' for float formatting, delimiter=',' for CSV
            np.savetxt(file_path, value, delimiter=',', fmt='%.8f')
            print(f"Saved {key}.csv to {output_folder_path}")
        else:
            print(f"Skipping saving for key '{key}' as it's not a NumPy array (type: {type(value)})")
    type_end_time = time.time()
    print(f"Finished parallel computation for data for config {config} in {type_end_time - type_start_time:.2f} seconds.")

In [4]:
# checked
D = {    #  configuration that yields the best validation accuracy
    "Configs": { 
                "P_diagonal_config" : np.asarray([0,3,1,2]),
                "P_vertical_config" : np.asarray([0,1,3,2]),
                "P_horizontal_config" : np.asarray([2,0,1,3])
              }
    }
D_configs = D["Configs"].values()


In [11]:
# Scaling data
scaled_sample = train_images/255.
# Encoding data using angle encoding
scaled_sample = scaled_sample * np.pi
from collections import defaultdict


for config in D_configs:
    # rearranging the input in respect to the configs
    input = scaled_sample[:,:,config]

    entropy_dict_for_symm = defaultdict(list)   # missing keys become []
    config = ''.join(map(str, config))
    process_input_type(input, config, entropy_dict_for_symm)
    # break

Processed 100/546 batches for 0312.
Processed 200/546 batches for 0312.
Processed 300/546 batches for 0312.
Processed 400/546 batches for 0312.
Processed 500/546 batches for 0312.
Created/Ensured directory: _OA_results_BreastMNIST_train_with_val/0312/Entropy
Saved Total_Avg_0312_Entropy_S1.csv to _OA_results_BreastMNIST_train_with_val/0312/Entropy
Saved Total_Avg_0312_Entropy_S2.csv to _OA_results_BreastMNIST_train_with_val/0312/Entropy
Saved Total_Max_0312_Entropy_S1.csv to _OA_results_BreastMNIST_train_with_val/0312/Entropy
Saved Total_Max_0312_Entropy_S2.csv to _OA_results_BreastMNIST_train_with_val/0312/Entropy
Saved S1_rho_0_0312_Entropy_S1_0.csv to _OA_results_BreastMNIST_train_with_val/0312/Entropy
Saved S1_rho_1_0312_Entropy_S1_1.csv to _OA_results_BreastMNIST_train_with_val/0312/Entropy
Saved S2_rho_0_0312_Entropy_S2_0.csv to _OA_results_BreastMNIST_train_with_val/0312/Entropy
Saved S2_rho_1_0312_Entropy_S2_1.csv to _OA_results_BreastMNIST_train_with_val/0312/Entropy
Saved Tot

We will focus only in the best 3 configs.

In [5]:
def open_load_entropy(symmetry, system, dataset_name="BreastMNIST"):
    
    system = str(system)
    file_path = f"/workspaces/QML-QPF/Entanglement_Filtering/BreastMNIST/_OA_results_{dataset_name}_train_with_val/{symmetry}/Entropy/S{system}_rho_0_{symmetry}_Entropy_S{system}_0.csv"
    S_rho0 = np.loadtxt(file_path, delimiter=',')
    return S_rho0
    

## Entanglement pooling and saving 

In [13]:
import joblib
import gc

def Entanglement_pooling(input, sample_size, threshold, S1_rho0, S2_rho0,Entanglement_pooled_file_name):
    """
    Function that will filter the output of the quantum circuit based on a threshold e.g. the mean of the entier dataset
    
    """
    # sample_size = reshaped_filtered_input.shape[0]  # number of images
    Entanglement_pooled_sample = list()
    # threshold = 0.106 # The mean of the entire dataset--will update it accordingly
    for i in range(sample_size):
        Entanglement_pooled_img = list()
        for j in range(S1_rho0.shape[-1]):
            #
            tmp_list = list()
            # Checking for S1
            if S1_rho0[i,j] < threshold: # keeping both qubits
                tmp_list.extend(input[i,j,:2]) 
            else: # Keeping only target qubit
                tmp_list.append(input[i,j,1])
                
            # Checking for S2
            if S2_rho0[i,j] < threshold: # keeping both qubits
                tmp_list.extend(input[i,j,2:]) 
            else: # Keeping only target qubit
                tmp_list.append(input[i,j,-1])
            # appending the pooled filtered input to the image
            Entanglement_pooled_img.append(tmp_list)
        # appending the pooled filtered image to the sample
        Entanglement_pooled_sample.append(Entanglement_pooled_img)
        
    joblib.dump(Entanglement_pooled_sample, f'{Entanglement_pooled_file_name}.joblib', compress=3)

    # free memory so the notebook stays fast
    del Entanglement_pooled_sample
    gc.collect()
        

In [6]:
import numpy as np

def Entntanglement_pool(input_data, symmetry, Entanglement_pooled_file_name):
    X = input_data.astype(np.float32, copy=False)   # (N,196,4)
    #S1_rho_0_0321_Entropy_S1 : S1_rho_0 = system 1;Entropy_S1 = qubit 1
    S1_rho0 = open_load_entropy(symmetry, 1)
    S2_rho0 = open_load_entropy(symmetry, 2)
    thr1 = S1_rho0.mean(axis=1, keepdims=True)  # (N,1)
    thr2 = S2_rho0.mean(axis=1, keepdims=True)  # (N,1)

    m1 = (S1_rho0 < thr1)  # (N,196)
    m2 = (S2_rho0 < thr2)  # (N,196)

    pooled = np.empty_like(X, dtype=np.float32)
    pooled[..., 0] = X[..., 0] * m1
    pooled[..., 1] = X[..., 1]
    pooled[..., 2] = X[..., 2] * m2
    pooled[..., 3] = X[..., 3]

    np.save(f"{Entanglement_pooled_file_name}.npy", pooled)
    # np.save(f"{Entanglement_pooled_file_name}_thr1.npy", thr1.squeeze().astype(np.float32))  # optional
    # np.save(f"{Entanglement_pooled_file_name}_thr2.npy", thr2.squeeze().astype(np.float32))  # optional


### Entanglement pooling the best 3 symmetries from previous experiment 

In [7]:
for _, value in D.items():
    for _, config in value.items():
        # print(symmetry)
        # filtered_input = train_layer.open(config)
        # reshaped_filtered_input = train_layer.transform(filtered_input)
        
        # input with specific symmetry/config to filter based on entropy level
        filtered_input = train_layer.open(config)
        filtered_input = train_layer.transform(filtered_input)
        
        symmetry = ''.join(map(str,config))
        
        # file with _ctr0 : replacing ctr with 0 when Entropy/Entanglement greater than threshold 
        # File namne to store the pooled sample
        Entanglement_pooled_file_name = f"{train_layer.name}/{symmetry}/Entanglement_pooled_{_[2:-7]}_{''.join(map(str,symmetry))}_ctr0"  
        
        # print(Entanglement_pooled_file_name)
        
        Entntanglement_pool(input_data=filtered_input,
                            symmetry=symmetry,\
                            Entanglement_pooled_file_name=Entanglement_pooled_file_name)
        # # Entanglement filtering
        # Entanglement_pooling(input=filtered_input, symmetry=symmetry,\
        #                     sample_size=filtered_input.shape[0],\
        #                     Entanglement_pooled_file_name=Entanglement_pooled_file_name)


## Trainign model


### Row-Level Padding

In [ ]:
import numpy as np

def pad_jagged_image(image_list, max_c=4):
    # Create a fixed-size buffer of zeros
    padded_img = np.zeros((len(image_list), max_c))
    
    for i, row in enumerate(image_list):
        # row is a 1D array of length 2, 3, or 4
        length = len(row)
        padded_img[i, :length] = row
        
    return padded_img


In [8]:
from keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau



def run(tr_images, label,  run_subfolder, folder_name, patience=15):
    log_dir = train_layer.name + f"/{label}" + f"/{folder_name}/run{run_subfolder}/"   
    # print(' log_dir ', log_dir)
    
    log_file =  log_dir + f"{folder_name}_{label}"
    # print(' log_file ', log_file)
    os.makedirs(log_dir, exist_ok=True)
    
    csv_logger = CSVLogger(log_file)
    
    
    # tensorboard_callback = keras.callbacks.TensorBoard(
    #     log_dir=log_file,
    #     histogram_freq=1,
    #     write_graph=True,
    #     write_images=True,
    #     write_steps_per_second=True,
    #     update_freq='batch',
    #     profile_batch=1,
    #     embeddings_freq=1,
    #     embeddings_metadata=None
    # )
    
    number_of_classes = len(np.unique(test_labels))
    # Chose which model you want to use depending on the number of hidden layers
    # Uncomment your model and run the code--pay attention to your folder/file naming to keep everything organized 
    ####---------------------------------------------------------------------####
    # 1 FC layer
    if run_subfolder == "_1FC":
        # simple ANN model with 0 hidden layers: 1FC
        q_model = keras.models.Sequential([
            keras.layers.Rescaling(scale=-1. / 127.5, offset=1),
            keras.layers.Flatten(),
            keras.layers.Dense(number_of_classes, activation="softmax")
        ])
    
    ####---------------------------------------------------------------------####
    
    # # 2 FC layer
    if run_subfolder == "_2FC":
    
        # simple ANN model with 1 hidden layers: 2FC
        q_model = keras.models.Sequential([
            keras.layers.Rescaling(scale=-1. / 127.5, offset=1),
            keras.layers.Flatten(),
            keras.layers.Dense(512, activation='relu'),
            keras.layers.Dense(number_of_classes, activation="softmax")
        ])
    
    ####---------------------------------------------------------------------####
    # # 3 FC layer
    if run_subfolder == "_3FC":
    
        # simple ANN model with 2 hidden layers: 3FC
        q_model = keras.models.Sequential([
            keras.layers.Rescaling(scale=-1. / 127.5, offset=1),
            keras.layers.Flatten(),
            keras.layers.Dense(512, activation='relu'),
            keras.layers.Dense(256, activation='relu'),
            keras.layers.Dense(number_of_classes, activation='softmax')
        ])
    
    ####---------------------------------------------------------------------####
    # # 4 FC layer
    if run_subfolder == "_4FC": 
        # simple ANN model with 3 hidden layers: 4FC
        q_model = keras.models.Sequential([
            keras.layers.Rescaling(scale=-1. / 127.5, offset=1),
            keras.layers.Flatten(),
            keras.layers.Dense(512, activation='relu'),
            keras.layers.Dense(256, activation='relu'),
            keras.layers.Dense(128, activation='relu'),
            keras.layers.Dense(number_of_classes, activation='softmax')
        ])
    
    ####---------------------------------------------------------------------####
    
    q_model.compile(
        optimizer='adam',
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    
    # 
    batch_size = 128
    epochs = 60
    
    
    # Stop training when a monitored metric has stopped improving.
    Early_stop = EarlyStopping(monitor="val_accuracy",
                                min_delta=0, # Minimum change in the monitored quantity to qualify as an improvement, i.e. an absolute change of less than min_delta, will count as no improvement.
                                patience=patience, # Number of epochs with no improvement after which training will be stopped.
                                verbose=1, # 1 displays messages when the callback takes an action
                                mode="max", # will stop when the quantity monitored has stopped increasing
                                baseline=None,
                                restore_best_weights=True, # to restore model weights from the epoch with the best value of the monitored quantity
                                start_from_epoch=0, #  Number of epochs to wait before starting to monitor improvement.
                            )

    
    # Callback to save the Keras model or model weights at some frequency.
    checkpoint_filepath = log_file  + 'model.keras'
    model_checkpoint_callback = ModelCheckpoint(filepath=checkpoint_filepath,
                                                monitor='val_accuracy',
                                                mode='max',
                                                verbose=1,
                                                save_best_only=True
                                                )
    # Reduce learning rate when a metric has stopped improving.
    reduce_lr_acc = ReduceLROnPlateau(monitor='val_accuracy',
                                        factor=0.1, # Float. Factor by which the learning rate will be reduced. new_lr = lr * factor.
                                        patience=10, # Integer. Number of epochs with no improvement after which learning rate will be reduced.
                                        verbose=1, # 1: update messages.
                                        mode="max", # 'max' mode it will be reduced when the quantity monitored has stopped increasing
                                        min_delta=0.0001, # Float. Threshold for measuring the new optimum, to only focus on significant changes.
                                        cooldown=0, #  Integer. Number of epochs to wait before resuming normal operation after the learning rate has been reduced.
                                        min_lr=0.0 #  Float. Lower bound on the learning rate.
                                    )


    
    q_history = q_model.fit(
        tr_images,
        train_labels,
        # validation_data=(te_images, test_labels),
        validation_split=0.2,
        batch_size=batch_size,
        epochs=epochs,
        verbose=2,
        # callbacks=[tensorboard_callback]
        callbacks = [Early_stop,csv_logger,model_checkpoint_callback, reduce_lr_acc]
    )

def model(variant, tr_layer):
    tr_images = Ent_pooled_data_padded_0s_array
    
    
    label = ''.join(map(str,Diagonal))


    run(tr_images, label)

In [9]:
run_subfolders = [f"_{str(i+1)}FC" for i in range(4)]

for _, value in D.items():
    for _, config in value.items():
        # print(symmetry)
        symmetry = ''.join(map(str,config))
        # File namne to store the pooled sample
        Entanglement_pooled_file_name = f"{train_layer.name}/{symmetry}/Entanglement_pooled_{_[2:-7]}_{symmetry}_ctr0" 
        # Too long to load: almost 5mins => think of other way probably to save and laod
        Entanglement_pooled_sample = np.load(f"{Entanglement_pooled_file_name}.npy") 
        approachs = ["Padding_0", "Entanglement_pool_summed", "No_Entanglement_pool_summed_avg"]
        for approach in approachs:
            # Padding with 0s
            if approach == "Padding_0":
                # 
                train_data =  Entanglement_pooled_sample
                # name of the folder for padded data
                folder_name = "Entanglement_pool_padded_0s"
            # Summing the entanglement pooled data
            if approach == "Entanglement_pool_summed":
                train_data = np.array([[sum(p)/len(p) for p in img] for img in Entanglement_pooled_sample])
                # name of the folder for padded data
                folder_name =  "Entanglement_pool_summed"
            
            # if approach == "No_Entanglement_pool_summed_avg":
            #     train_data = train_layer.open(config)
            #     train_data = train_data.mean(axis=-1)    # shape: (N, H, W)
            #     # name 
            #     folder_name =  "No_Entanglement_pool_summed_avg"
                
            for run_subfolder in run_subfolders:
                run(tr_images=train_data, label=symmetry,\
                    run_subfolder=run_subfolder, folder_name=folder_name)
                
        

Epoch 1/60

Epoch 1: val_accuracy improved from -inf to 0.68182, saving model to _OA_results_BreastMNIST_train_with_val/0312/Entanglement_pool_padded_0s/run_1FC/Entanglement_pool_padded_0s_0312model.keras
4/4 - 1s - loss: 0.9067 - accuracy: 0.4564 - val_loss: 0.6659 - val_accuracy: 0.6818 - lr: 0.0010 - 884ms/epoch - 221ms/step
Epoch 2/60

Epoch 2: val_accuracy improved from 0.68182 to 0.73636, saving model to _OA_results_BreastMNIST_train_with_val/0312/Entanglement_pool_padded_0s/run_1FC/Entanglement_pool_padded_0s_0312model.keras
4/4 - 0s - loss: 0.7134 - accuracy: 0.7248 - val_loss: 0.7490 - val_accuracy: 0.7364 - lr: 0.0010 - 74ms/epoch - 18ms/step
Epoch 3/60

Epoch 3: val_accuracy did not improve from 0.73636
4/4 - 0s - loss: 0.6939 - accuracy: 0.7294 - val_loss: 0.6465 - val_accuracy: 0.6636 - lr: 0.0010 - 37ms/epoch - 9ms/step
Epoch 4/60

Epoch 4: val_accuracy did not improve from 0.73636
4/4 - 0s - loss: 0.5936 - accuracy: 0.6927 - val_loss: 0.7169 - val_accuracy: 0.5909 - lr: 

In [19]:
symmetry = np.asarray([0,3,1,2])
symmetry = ''.join(map(str,config))
Entanglement_pooled_file_name = f"{train_layer.name}/{symmetry}/Entanglement_pooled_{_[2:-7]}_{symmetry}" 
# Too long to load: almost 5mins => think of other way probably to save and laod
Entanglement_pooled_sample = joblib.load(f'{Entanglement_pooled_file_name}.joblib') 

In [22]:
len(Entanglement_pooled_sample[0])

196

In [23]:
train_images.shape

(546, 196, 4)